In [11]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [12]:
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

from src.datasets import FaceDataset, train_transform, val_transform, FaceTestDataset
from src.model_upgraded import ArtifactAwareCNN
from src.dual_branch_model import DualBranchMonster
from src.engine import train_one_epoch, evaluate
from src.utils import find_best_threshold

In [13]:
DATA_DIR = Path("dataset")
TRAIN_DIR = DATA_DIR / "train_images"
TRAIN_CSV = DATA_DIR / "train_solution.csv"
TEST_DIR = DATA_DIR / "test_images"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

cuda
NVIDIA GeForce RTX 3080


In [14]:
df = pd.read_csv(TRAIN_CSV, header=None, names=["Id", "target"])
df["path"] = df["Id"].apply(lambda x: TRAIN_DIR / f"{x}.jpg")

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["target"]
)

train_df = train_df.sample(min(1000, len(train_df)), random_state=42)
val_df = val_df.sample(min(200, len(val_df)), random_state=42)

print(len(train_df), len(val_df))

1000 200


In [15]:
train_dataset = FaceDataset(train_df, transform=train_transform)
val_dataset = FaceDataset(val_df, transform=val_transform)

torch.backends.cudnn.enabled = False
torch.backends.cudnn.benchmark = False

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

In [16]:
# model = ArtifactAwareCNN().to(device)
model = DualBranchMonster(base_ch=32, blocks=(2, 2, 3, 3)).to(device)
num_pos = (train_df["target"] == 1).sum()
num_neg = (train_df["target"] == 0).sum()
pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32, device=device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

In [17]:
import time
import torch

model = model.to(device)
model.train()

x, y = next(iter(train_loader))
x = x.to(device)
y = y.to(device).float().view(-1, 1)

if device.type == "cuda":
    torch.cuda.synchronize()

t0 = time.time()
out = model(x)
loss = criterion(out, y)
if device.type == "cuda":
    torch.cuda.synchronize()
t1 = time.time()

loss.backward()
if device.type == "cuda":
    torch.cuda.synchronize()
t2 = time.time()

optimizer.step()
optimizer.zero_grad(set_to_none=True)
if device.type == "cuda":
    torch.cuda.synchronize()
t3 = time.time()

print(f"forward+loss: {t1-t0:.3f}s")
print(f"backward:     {t2-t1:.3f}s")
print(f"step:         {t3-t2:.3f}s")

forward+loss: 0.326s
backward:     0.442s
step:         0.066s


In [18]:
import copy
from tqdm.notebook import tqdm
best_val_f1 = -1
best_threshold = 0.5
best_state = None

n_epochs = 5

for epoch in range(n_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_f1_05, val_probs, val_labels = evaluate(
        model, val_loader, criterion, device, threshold=0.5
    )

    thr, val_f1_best = find_best_threshold(val_labels, val_probs)

    tqdm.write(
        f"Epoch {epoch+1}/{n_epochs} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_f1@0.5={val_f1_05:.4f} | "
        f"best_thr={thr:.2f} | "
        f"best_val_f1={val_f1_best:.4f}"
    )

    if val_f1_best > best_val_f1:
        best_val_f1 = val_f1_best
        best_threshold = thr
        best_state = copy.deepcopy(model.state_dict())

    scheduler.step(val_f1_best)

Epoch 1/5 | train_loss=1.3345 | val_loss=1.3007 | val_f1@0.5=0.3787 | best_thr=0.51 | best_val_f1=0.4118


Epoch 2/5 | train_loss=1.1753 | val_loss=2.4540 | val_f1@0.5=0.0400 | best_thr=0.07 | best_val_f1=0.4086


Epoch 3/5 | train_loss=1.1628 | val_loss=1.3087 | val_f1@0.5=0.3188 | best_thr=0.49 | best_val_f1=0.4598


Epoch 4/5 | train_loss=1.1718 | val_loss=1.3243 | val_f1@0.5=0.3680 | best_thr=0.46 | best_val_f1=0.4086


Epoch 5/5 | train_loss=1.1573 | val_loss=1.2755 | val_f1@0.5=0.3740 | best_thr=0.57 | best_val_f1=0.4173


In [19]:
model.load_state_dict(best_state)
print("Best F1:", best_val_f1)
print("Best threshold:", best_threshold)

Best F1: 0.45977011494252873
Best threshold: 0.49000000000000005


In [20]:
from torchvision import transforms
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    ),
])

In [21]:
test_dataset = FaceTestDataset(TEST_DIR, transform=test_transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4
)

print("Test size:", len(test_dataset))

Test size: 10000


In [22]:
from tqdm import tqdm

@torch.no_grad()
def predict_test(model, loader, device, threshold):
    model.eval()

    ids = []
    preds = []

    for images, image_ids in tqdm(loader, desc="Predicting"):
        images = images.to(device)

        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy().ravel()
        batch_preds = (probs >= threshold).astype(int)

        ids.extend(image_ids.numpy().tolist())
        preds.extend(batch_preds.tolist())

    return ids, preds

In [ ]:
test_ids, test_preds = predict_test(
    model=model,
    loader=test_loader,
    device=device,
    threshold=best_threshold
)

print(test_ids[:10])
print(test_preds[:10])
print(len(test_ids), len(test_preds))

Predicting:   4%|▍         | 7/157 [00:36<06:40,  2.67s/it]  

In [ ]:
submission = pd.DataFrame({
    "id": test_ids,
    "target_feature": test_preds
}).sort_values("id").reset_index(drop=True)

submission.head()
submission.to_csv("submission_new.csv", index=False)
print("submission_new.csv saved")

submission_new.csv saved
